In [1]:
import pandas as pd
import numpy as np

# ==============================
# 1️⃣ LOAD DATA
# ==============================

DATA_PATH = "data/"

appt_w1 = pd.read_csv(DATA_PATH + "AppointmentDataWeek1.csv")
appt_w2 = pd.read_csv(DATA_PATH + "AppointmentDataWeek2.csv")

prov_w1 = pd.read_csv(DATA_PATH + "ProviderRoomAssignmentWeek1.csv")
prov_w2 = pd.read_csv(DATA_PATH + "ProviderRoomAssignmentWeek2.csv")

room_dist = pd.read_csv(DATA_PATH + "room_proximity_matrix.csv", index_col=0)

# ==============================
# 2️⃣ APPOINTMENT TABLE
# ==============================

def clean_appointments(df, week_label):
    df = df.copy()
    df["Week"] = week_label
    
    # Standardize column names (edit if needed)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    
    # Ensure duration numeric
    if "duration" in df.columns:
        df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
    
    return df

appt_w1_clean = clean_appointments(appt_w1, "Week1")
appt_w2_clean = clean_appointments(appt_w2, "Week2")

appointment_table = pd.concat([appt_w1_clean, appt_w2_clean], ignore_index=True)

print("Appointment Table Created")
display(appointment_table.head())

Appointment Table Created


,patient_id,appt_date,primary_provider,apptstatussingleview,cancelled_appts,deleted_appts,no_show_appts,appt_time,appt_duration,appt_type,week
0,Patient 01,11-10-2025,HPW101,Finished,N,N,N,09:30:00,5,NaN,Week1
1,Patient 02,11-10-2025,HPW101,NaN,N,Y,N,09:45:00,5,NaN,Week1
2,Patient 03,11-10-2025,HPW101,Finished,N,N,N,10:25:00,5,NaN,Week1
3,Patient 04,11-10-2025,HPW101,Finished,N,N,N,14:55:00,15,NaN,Week1
4,Patient 05,11-10-2025,HPW101,Finished,N,N,N,09:55:00,5,NaN,Week1


In [2]:
# ==============================
# PROVIDER AVAILABILITY MATRIX
# ==============================

def build_provider_availability(df, week_label):
    df = df.copy()
    df["Week"] = week_label
    
    df.columns = [c.strip() for c in df.columns]
    
    day_cols = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
    
    records = []
    
    for _, row in df.iterrows():
        provider = row["Primary Provider"]
        
        for day in day_cols:
            room = row[day]
            
            if pd.isna(room) or "N/A" in str(room) or "CLOSED" in str(room) or "NO ROOM" in str(room):
                available = 0
                room_assigned = None
            else:
                available = 1
                room_assigned = room
            
            records.append({
                "Provider": provider,
                "Day": day,
                "Week": week_label,
                "Available": available,
                "AssignedRoom": room_assigned
            })
    
    return pd.DataFrame(records)

prov_avail_w1 = build_provider_availability(prov_w1, "Week1")
prov_avail_w2 = build_provider_availability(prov_w2, "Week2")

provider_availability = pd.concat([prov_avail_w1, prov_avail_w2], ignore_index=True)

print("Provider Availability Matrix Created")
display(provider_availability.head())

Provider Availability Matrix Created


,Provider,Day,Week,Available,AssignedRoom
0,HPW101,Monday,Week1,1,Room 15
1,HPW101,Tuesday,Week1,0,None
2,HPW101,Wednesday,Week1,0,None
3,HPW101,Thursday,Week1,0,None
4,HPW101,Friday,Week1,1,RM 5 (AM)


In [3]:
# ==============================
# ROOM DISTANCE MATRIX
# ==============================

room_distance_matrix = room_dist.copy()

# Ensure numeric
room_distance_matrix = room_distance_matrix.apply(pd.to_numeric)

print("Room Distance Matrix Created")
display(room_distance_matrix.head())

Room Distance Matrix Created


,ER1,ER2,ER3,ER4,ER5,ER6,ER7,ER8,ER9,ER10,ER11,ER12,ER13,ER14,ER15,ER16
Room,,,,,,,,,,,,,,,,
ER1,0.0,0.5,1.5,2.5,4.0,4.7,2.0,3.0,4.7,4.1,3.3,1.5,3.0,6.1,11.5,12.0
ER2,0.5,0.0,1.0,2.0,3.5,4.2,1.5,2.5,4.2,3.6,2.8,2.0,3.5,5.6,11.0,11.5
ER3,1.5,1.0,0.0,1.0,2.5,3.2,0.5,1.5,3.2,2.6,1.8,3.0,4.5,4.6,10.0,10.5
ER4,2.5,2.0,1.0,0.0,1.5,2.2,1.5,1.5,2.2,1.6,2.8,4.0,5.5,3.6,9.0,9.5
ER5,4.0,3.5,2.5,1.5,0.0,0.7,3.0,3.0,2.5,3.1,4.3,5.5,6.0,3.9,7.5,8.0


In [5]:
room_distance_long = (
    room_distance_matrix
        .stack()
        .reset_index()
        .rename(columns={
            "level_0": "Room_i",
            "level_1": "Room_j",
            0: "Distance"
        })
)

room_distance_long.to_csv("data/processed_room_distances_long.csv", index=False)

room_distance_long.head()

,Room,Room_j,Distance
0,ER1,ER1,0.0
1,ER1,ER2,0.5
2,ER1,ER3,1.5
3,ER1,ER4,2.5
4,ER1,ER5,4.0


In [6]:
appointment_table.to_csv("data/processed_appointments.csv", index=False)
provider_availability.to_csv("data/processed_provider_availability.csv", index=False)
room_distance_long.to_csv("data/processed_room_distances_long.csv", index=False)